In [ ]:
# default_exp models.naive2.naive2

In [ ]:
#hide
%load_ext autoreload
%autoreload 2

In [ ]:
#export
import numpy as np
from numpy.random import seed
seed(1)

import pandas as pd
from math import sqrt

# Naive Benchmark Model

> API details.

In [ ]:
# export
class Naive:
    """
    Naive model.
    This benchmark model produces a forecast that is equal to
    the last observed value for a given time series.
    """
    def __init__(self):
        pass
    
    def fit(self, ts_init):
        """
        ts_init: the original time series
        ts_naive: last observations of time series
        """
        self.ts_naive = [ts_init[-1]]
        return self

    def predict(self, h):
        return np.array(self.ts_naive * h)


# Seasonal Naive Benchmark Model

> API details.

In [ ]:
# export
class SeasonalNaive:
    """
    Seasonal Naive model.
    This benchmark model produces a forecast that is equal to
    the last observed value of the same season for a given time 
    series.
    """
    def __init__(self):
        pass
    
    def fit(self, ts_init, seasonality):
        """
        ts_init: the original time series
        frcy: frequency of the time series
        ts_naive: last observations of time series
        """
        self.ts_seasonal_naive = ts_init[-seasonality:]
        return self

    def predict(self, h):
        repetitions = int(np.ceil(h/len(self.ts_seasonal_naive)))
        y_hat = np.tile(self.ts_seasonal_naive, reps=repetitions)[:h]
        return y_hat

# Naive2 Benchmark Model

> API details.

The Naive2 benchmark model, as explained in the [M4 guide](https://www.m4.unic.ac.cy/wp-content/uploads/2018/03/M4-Competitors-Guide.pdf) is determined by the following process:
- Test the time-series for seasonal patterns by performing a 90% auto-correlation.
- Based on the test select between a Naive benchmark (random walk) or a spline detrended seasonal forecast.

In [ ]:
#export
def detrend(insample_data):
    """
    Calculates a & b parameters of LRL
    :param insample_data:
    :return:
    """
    x = np.arange(len(insample_data))
    a, b = np.polyfit(x, insample_data, 1)
    return a, b

def deseasonalize(original_ts, ppy):
    """
    Calculates and returns seasonal indices
    :param original_ts: original data
    :param ppy: periods per year
    :return:
    """
    """
    # === get in-sample data
    original_ts = original_ts[:-out_of_sample]
    """
    if seasonality_test(original_ts, ppy):
        # ==== get moving averages
        ma_ts = moving_averages(original_ts, ppy)

        # ==== get seasonality indices
        le_ts = original_ts * 100 / ma_ts
        le_ts = np.hstack((le_ts, np.full((ppy - (len(le_ts) % ppy)), np.nan)))
        le_ts = np.reshape(le_ts, (-1, ppy))
        si = np.nanmean(le_ts, 0)
        norm = np.sum(si) / (ppy * 100)
        si = si / norm
    else:
        si = np.ones(ppy)

    return si

def moving_averages(ts_init, window):
    """
    Calculates the moving averages for a given TS
    :param ts_init: the original time series
    :param window: window length
    :return: moving averages ts
    """
    """
    As noted by Professor Isidro Lloret Galiana:
    line 82:
    if len(ts_init) % 2 == 0:
    
    should be changed to
    if window % 2 == 0:
    
    This change has a minor (less then 0.05%) impact on the calculations of the seasonal indices
    In order for the results to be fully replicable this change is not incorporated into the code below
    """
    ts_init = pd.Series(ts_init)
    
    if len(ts_init) % 2 == 0:
        ts_ma = ts_init.rolling(window, center=True).mean()
        ts_ma = ts_ma.rolling(2, center=True).mean()
        ts_ma = np.roll(ts_ma, -1)
    else:
        ts_ma = ts_init.rolling(window, center=True).mean()

    return ts_ma

def seasonality_test(original_ts, ppy):
    """
    Seasonality test
    :param original_ts: time series
    :param ppy: periods per year
    :return: boolean value: whether the TS is seasonal
    """
    s = acf(original_ts, 1)
    for i in range(2, ppy):
        s = s + (acf(original_ts, i) ** 2)

    limit = 1.645 * (sqrt((1 + 2 * s) / len(original_ts)))

    return (abs(acf(original_ts, ppy))) > limit

def acf(data, k):
    """
    Autocorrelation function
    :param data: time series
    :param k: lag
    :return:
    """
    m = np.mean(data)
    s1 = 0
    for i in range(k, len(data)):
        s1 = s1 + ((data[i] - m) * (data[i - k] - m))

    s2 = 0
    for i in range(0, len(data)):
        s2 = s2 + ((data[i] - m) ** 2)

    return float(s1 / s2)

In [ ]:
# export
class Naive2:
    """
    Naive2 model.
    Popular benchmark model for time series forecasting that automatically adapts
    to the potential seasonality of a series based on an autocorrelation test.
    If the series is seasonal the model composes the predictions of Naive and SeasonalNaive,
    else the model predicts on the simple Naive.
    """
    def __init__(self, seasonality):
        self.seasonality = seasonality
        
    def fit(self, ts_init):
        seasonality_in = deseasonalize(ts_init, ppy=self.seasonality)
        windows = int(np.ceil(len(ts_init) / self.seasonality))
        
        self.ts_init = ts_init
        self.s_hat = np.tile(seasonality_in, reps=windows)[:len(ts_init)]
        self.ts_des = ts_init / self.s_hat
                        
        return self
        
    def predict(self, h):
        s_hat = SeasonalNaive().fit(self.s_hat, 
                                    seasonality=self.seasonality).predict(h)
        r_hat = Naive().fit(self.ts_des).predict(h)              
        y_hat = s_hat * r_hat
        return y_hat

# Naive Benchmark Usage Example

In [ ]:
import pandas as pd
from nixtla.data.datasets.epf import EPF, EPFInfo
from nixtla.data.tsloader import TimeSeriesLoader

import pylab as plt
from pylab import rcParams
plt.style.use('seaborn-whitegrid')
plt.rcParams['font.family'] = 'serif'

FONTSIZE = 17

# Load and plot data
Y_df, X_df, S_df = EPF.load_groups(directory='./data', groups=['NP'])

fig = plt.figure(figsize=(15, 6))
plt.plot(Y_df.ds, Y_df.y.values, color='#628793', linewidth=0.4)
plt.ylabel('Price [EUR/MWh]', fontsize=15)
plt.xlabel('Date', fontsize=15)
# plt.savefig('./results/NP.png', bbox_inches = 'tight')
plt.show()

In [ ]:
from itertools import chain
from multiprocessing import Pool
from numpy.lib.stride_tricks import sliding_window_view
from nixtla.losses.numpy import mae

def fit_predict_naive2(Y, output_size=24, seasonality=24):
    y_hat_list = []
    for idx in range(len(Y)):
        y_id = Y[idx, :]
        y_hat_id = Naive2(seasonality).fit(y_id).predict(output_size)
        y_hat_list += [y_hat_id]
    return y_hat_list

def parallelized_naive2(Y, n_cores=32):
    # Y
    n_partitions = n_cores**3//4  # number of partitions to split the array
    Y_split  = np.array_split(Y, n_partitions)
    pool = Pool(n_cores)
    Y_hat_list = list(chain.from_iterable(pool.map(fit_predict_naive2, Y_split)))
    pool.close()
    pool.join()
    Y_hat = np.vstack(Y_hat_list)
    return Y_hat

In [ ]:
Y_windows = sliding_window_view(Y_df.y.values, window_shape=24*7)
day_idxs = (np.arange(len(Y_windows)) % 24) == 0
Y_windows = Y_windows[day_idxs, :]
print("Y_windows.shape", Y_windows.shape)

In [ ]:
Y_hat = parallelized_naive2(Y=Y_windows)
Y_hat = Y_hat.reshape(-1)
Y_hat.shape

In [ ]:
fig = plt.figure(figsize=(15, 6))
plt.plot(Y_df.ds[168:], Y_df.y.values[168:], color='#628793', linewidth=0.4, label='true')
plt.plot(Y_df.ds[168:], Y_hat[:len(Y_hat)-24], color='peru', linewidth=0.4, label='forecast')
plt.ylabel('Price [EUR/MWh]', fontsize=15)
plt.xlabel('Date', fontsize=15)
plt.legend()
# plt.savefig('./results/NP.png', bbox_inches = 'tight')
plt.show()

In [ ]:
naive2_mae = mae(y=Y_df.y.values[168:], y_hat=Y_hat[:len(Y_hat)-24])
print(f'MAE: {naive2_mae}')